# 03 — Module-level structure of MOM6 + FMS2

Zooming out from routines to **modules**:
`ParseForest.get_module_dependency_graph()` returns a NetworkX digraph whose
nodes are module names and whose edges are USE relationships (lifted from any
contained scope to the enclosing module) plus derived-type EXTENDS
relationships (a child type depends on the module defining its parent).

Analyses in this notebook: fan-in/fan-out, entry points and foundations,
acyclicity as an invariant check, partitioning modules by subproject,
**which parts of FMS2 MOM6 actually needs** (and which it doesn't), the direct
API surface between the two, type-extension coupling, and an inline
ipycytoscape rendering of one module's neighbourhood.

Prerequisites: notebook 01 for concepts; corpus access as in notebook 02
(`GROUNDLINE_CORPUS` overrides the glade default).

## Parameters

In [ ]:
import os
from pathlib import Path

CORPUS = Path(os.environ.get(
    "GROUNDLINE_CORPUS",
    "/glade/work/altuntas/turbo-stack/bin/flang_ptree/MOM6_using_FMS2",
))
# Module whose neighbourhood the final section renders.
FOCUS_MODULE = os.environ.get("GROUNDLINE_FOCUS_MODULE", "mom_diag_mediator")
# Cap for the rendered neighbourhood (largest-degree neighbours kept).
RENDER_CAP = 30

assert CORPUS.is_dir(), (
    f"{CORPUS} not found — set GROUNDLINE_CORPUS to a directory of *_ptree dumps."
)
paths = sorted(p for p in CORPUS.rglob("*_ptree") if p.is_file())
print(f"{len(paths)} parse-tree dumps under {CORPUS}")

## Build the forest and the module graph (~40 s)

Note the graph also contains modules the corpus *references but never defines*
(`netcdf`, `mpi`, `iso_c_binding`, …) — they appear as edge targets, mirroring
how unresolved call targets are first-class in the call relation.

In [ ]:
import networkx as nx

from groundline.parse_forest import ParseForest

forest = ParseForest(paths)
ir = forest.ir
G = forest.get_module_dependency_graph()

defined = {m.name for m in ir.modules if m.defined}
external = sorted(set(G.nodes) - defined)
print(f"{G.number_of_nodes()} modules, {G.number_of_edges()} dependency edges")
print(f"{len(external)} referenced-but-undefined modules: {external}")

## Fan-in and fan-out

**Fan-in** (in-degree): how many modules depend on this one — the load-bearing
infrastructure; touching these has the widest blast radius. **Fan-out**
(out-degree): how many modules this one depends on — the big integrators
sitting near the top of the stack.

In [ ]:
print("most depended-upon (fan-in):")
for name, deg in sorted(G.in_degree(), key=lambda x: -x[1])[:10]:
    print(f"  {deg:4}  {name}")
print("\nbiggest dependers (fan-out):")
for name, deg in sorted(G.out_degree(), key=lambda x: -x[1])[:10]:
    print(f"  {deg:4}  {name}")

## Entry points and foundations

Modules **nobody uses** (in-degree 0) are entry points or dead weight — top-level
drivers, or library features the rest of the corpus never touches. Modules that
**use nothing** (out-degree 0) are the foundations (`platform_mod`,
kind-definition modules, …).

In [ ]:
roots = sorted(n for n, d in G.in_degree() if d == 0)
leaves = sorted(n for n, d in G.out_degree() if d == 0)
print(f"{len(roots)} unused-by-anyone modules (entry points / candidates for pruning):")
print(" ", roots)
print(f"\n{len(leaves)} modules with no dependencies (foundations):")
print(" ", leaves)

## Acyclicity — a first invariant check

Fortran forbids circular module USE, so the USE half of this graph *must* be
acyclic — checking it is a tiny example of the invariant style groundline is
heading toward (Phase 5): state a property, evaluate it over the fact graph,
and demand the witnesses when it fails.

The graph as a whole is allowed one kind of "cycle": a **self-loop**, e.g. a
derived type extending a parent type defined in its own module.

In [ ]:
self_loops = sorted(nx.nodes_with_selfloops(G))
sccs = [c for c in nx.strongly_connected_components(G) if len(c) > 1]
print(f"self-loops: {len(self_loops)}  e.g. {self_loops[:5]}")
print(f"multi-module cycles (must be none): {sccs}")
assert not sccs, "circular module dependency — should be impossible in legal Fortran"

## Partitioning modules by subproject

Which modules belong to FMS2, and which to MOM6? **The IR deliberately carries
no source-file provenance** — entities are language-level facts, and "which
repository a file lives in" is not one of them (the pre-seam notebooks leaned on
a `parse_tree_path` node attribute that no longer exists). This is a real gap
for corpus-level analyses; it is recorded as a finding rather than papered over
with a frontend import.

The seam-respecting workaround: the corpus directory layout *is* the
provenance. Extract each subdirectory on its own and note which modules it
defines. (Costs one extra pass over the corpus, ~40 s.)

In [ ]:
project_of = {}
for subdir in sorted(p for p in CORPUS.iterdir() if p.is_dir()):
    sub_ir = ParseForest(sorted(subdir.rglob("*_ptree"))).ir
    for m in sub_ir.modules:
        if m.defined:
            project_of.setdefault(m.name, subdir.name)

from collections import Counter
print(Counter(project_of.values()))

mom6_modules = {m for m, p in project_of.items() if p in ("MOM6", "MOM6-infra")}
fms_modules = {m for m, p in project_of.items() if p == "FMS2"}

## Which parts of FMS2 does MOM6 actually need?

Transitive closure over the dependency edges: everything reachable from any
MOM6 module. FMS2 modules *outside* that closure are dead weight for a
MOM6-only build — the kind of fact a dependency-pruning or porting effort
starts from. (This rebuilds the core analysis of the retired
`module_dependency*.ipynb` notebooks on the post-seam API.)

In [ ]:
reachable = set()
for m in mom6_modules:
    if m in G:
        reachable |= nx.descendants(G, m)

fms_needed = sorted(reachable & fms_modules)
fms_unused = sorted(fms_modules - reachable)
print(f"FMS2 modules: {len(fms_modules)}  "
      f"needed by MOM6: {len(fms_needed)}  not needed: {len(fms_unused)}")
print("\nnot needed by MOM6:")
print(fms_unused)

### The direct API surface

Restricting to *direct* edges answers a different question: which FMS2 modules
does MOM6 code name in a USE statement? This is the actual coupling surface —
the modules an infrastructure swap (FMS2 → TIM) has to re-provide.

In [ ]:
direct_users = {}
for m in sorted(mom6_modules):
    if m not in G:
        continue
    for dep in set(G.successors(m)) & fms_modules:
        direct_users.setdefault(dep, []).append(m)

print(f"{len(direct_users)} FMS2 modules are directly USE'd by MOM6 code:\n")
for dep, users in sorted(direct_users.items(), key=lambda kv: -len(kv[1])):
    print(f"  {len(users):3} MOM6 module(s) use {dep}")

## Type-extension coupling

EXTENDS relationships come straight from the IR: each derived type records its
parent type name. Cross-module extensions are a stronger coupling than USE —
the child inherits representation and bindings, not just names.

In [ ]:
from groundline.ir import MODULE


def enclosing_module(eid):
    cur = ir.get(eid)
    while cur is not None and cur.kind != MODULE:
        cur = ir.get(cur.scope) if cur.scope else None
    return cur.name if cur else None


type_module = {dt.name.lower(): enclosing_module(dt.id) for dt in ir.derived_types}

extensions = [(dt, dt.parent_type) for dt in ir.derived_types if dt.parent_type]
cross = [(dt, parent) for dt, parent in extensions
         if type_module.get(parent.lower()) not in (None, enclosing_module(dt.id))]
print(f"{len(extensions)} derived types extend a parent; "
      f"{len(cross)} cross a module boundary:\n")
for dt, parent in sorted(cross, key=lambda x: x[0].id):
    print(f"  {dt.id}  EXTENDS  {type_module[parent.lower()]}::{parent}")

## Rendering one module's neighbourhood

The full graph (427 nodes, ~3200 edges) is unreadable as a picture; a single
module's direct neighbourhood is not. ipycytoscape (the same library behind the
Explorer) renders a NetworkX graph inline. Blue = the focus module, red = its
dependencies (successors), grey = its dependents (predecessors, trimmed to the
highest-degree ones when there are too many).

In [ ]:
import ipycytoscape

deps = sorted(G.successors(FOCUS_MODULE))
dependents = sorted(G.predecessors(FOCUS_MODULE), key=lambda n: -G.degree(n))
kept_dependents = dependents[: max(0, RENDER_CAP - len(deps))]
if len(dependents) > len(kept_dependents):
    print(f"({len(dependents)} dependents; showing the {len(kept_dependents)} "
          f"highest-degree ones)")

neighbourhood = G.subgraph([FOCUS_MODULE, *deps, *kept_dependents]).copy()
for n in neighbourhood.nodes:
    neighbourhood.nodes[n]["role"] = (
        "focus" if n == FOCUS_MODULE else "dep" if n in deps else "dependent"
    )

widget = ipycytoscape.CytoscapeWidget()
widget.graph.add_graph_from_networkx(neighbourhood, directed=True)
widget.set_style([
    {"selector": "node", "style": {
        "label": "data(id)", "font-size": "9px", "background-color": "#95a5a6",
        "width": "14px", "height": "14px"}},
    {"selector": 'node[role = "focus"]', "style": {
        "background-color": "#3498db", "width": "24px", "height": "24px",
        "font-weight": "bold"}},
    {"selector": 'node[role = "dep"]', "style": {"background-color": "#e74c3c"}},
    {"selector": "edge", "style": {
        "curve-style": "bezier", "target-arrow-shape": "triangle",
        "width": 1, "line-color": "#cccccc", "target-arrow-color": "#cccccc"}},
])
widget.set_layout(name="cose")
widget

## Where to go next

* **04_confidence_queries** — the same corpus, queried through the confidence
  strata: what is certain, what is guessed, and what that means for invariants.

**Recorded finding** (see the subproject section): corpus-level analyses want a
source-provenance fact (which file/tree defined this entity) that the IR does
not carry; the per-subdirectory extraction above is the seam-respecting
workaround until a deliberate IR decision is made.